<a href="https://colab.research.google.com/github/1706712022/etl-data-pipeline1706712022/blob/main/Bodegas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

PARCIAL 2 LIMPIEZA DE DATOS

[GITHUB](https://github.com/1706712022/etl-data-pipeline1706712022.git)

In [52]:
import pandas as pd

In [53]:
url = "https://raw.githubusercontent.com/1706712022/etl-data-pipeline1706712022/refs/heads/main/data/raw/E_bodegas.csv"

In [54]:
df_bodegas = pd.read_csv(url)
print("Dataset Cargado Correctamente")

Dataset Cargado Correctamente


EXPLORACION DE DATOS

In [55]:
df_bodegas.head()

,id_bodega,bodega,ubicacion,capacidad_m2
0,BOD100,Central 0,Usulután,1292 m2
1,BOD101,Sur 1,San Miguel,2047
2,BOD102,Central 2,Sonsonate,651 m2
3,BOD103,Occidente 3,San Miguel,2250
4,BOD104,Sur 4,Santa Ana,239


In [56]:
print("Columnas del dataset");
print(df_bodegas.columns);

Columnas del dataset
Index(['id_bodega', 'bodega', 'ubicacion', 'capacidad_m2'], dtype='object')


In [57]:
print("Tipos de datos del dataset");
print(df.dtypes);

Tipos de datos del dataset
id_bodega       object
bodega          object
ubicacion       object
capacidad_m2    object
dtype: object


In [58]:
print("Valores nulos del dataset");
print(df.isnull().sum());

Valores nulos del dataset
id_bodega       2
bodega          0
ubicacion       0
capacidad_m2    0
dtype: int64


In [59]:
print ("Valores duplicados del dataset");
print (df.duplicated().sum());

Valores duplicados del dataset
2


NORMALIZACION DE DATOS

In [60]:
#Quitar espacios en blanco al inicio y final de todas las celdas de texto
df_bodegas = df_bodegas.apply(lambda x: x.str.strip() if x.dtype == "object" else x)

In [61]:
#Normalizacion de nombres en la bodegas (Primer letra Mayuscula)
df_bodegas['bodega'] = df_bodegas['bodega'].str.title()

In [62]:
#Normalizacion de nombres en ubicaciones (Primer letra Mayuscula)
df_bodegas['ubicacion'] = df_bodegas['ubicacion'].str.title()

In [63]:
#Dejar solo el numero de la capacidad por bodega
df_bodegas['capacidad_m2'] = df_bodegas['capacidad_m2'].str.replace(' m2', '', case=False)
df_bodegas['capacidad_m2'] = pd.to_numeric(df_bodegas['capacidad_m2'], errors='coerce')

VALIDACION Y SEPARACION DE DATOS : REJECTED / CURATED

In [64]:
# Creamos la columna para el reporte de errores
df_bodegas['motivo_rechazo'] = ""

In [65]:
## Regla 1: ID de bodega nulo
if 'motivo_rechazo' not in df_bodegas.columns:
    df_bodegas['motivo_rechazo'] = ""
df_bodegas.loc[df_bodegas['id_bodega'].isna(), 'motivo_rechazo'] += "ID de bodega nulo; "

In [66]:
# Regla 2: Capacidad inválida o nula
df_bodegas.loc[df_bodegas['capacidad_m2'].isna(), 'motivo_rechazo'] += "Capacidad no numérica; "

In [67]:
# Regla 3: Detectar duplicados (por ID de bodega)
duplicados = df_bodegas.duplicated(subset=['id_bodega'], keep='first')
df_bodegas.loc[duplicados & (df_bodegas['id_bodega'].notna()), 'motivo_rechazo'] += "ID de bodega duplicado; "

In [68]:
#SEPARACIÓN
bodegas_curated = df_bodegas[df_bodegas['motivo_rechazo'] == ""].copy()
bodegas_rejects = df_bodegas[df_bodegas['motivo_rechazo'] != ""].copy()

In [69]:
# Limpiamos las columnas para el archivo final Curated
bodegas_curated = bodegas_curated.drop(columns=['motivo_rechazo'])

In [72]:
#VERIFICACION DE DATOS
print("datos limpios",bodegas_curated.shape)
print("datos con errores",bodegas_rejects.shape)

datos limpios (16, 4)
datos con errores (4, 5)


CARGA DE DATOS A PostgreSQL

In [76]:
import os

# Crear las carpetas si no existen
os.makedirs('curated', exist_ok=True)
os.makedirs('rejects', exist_ok=True)

# Guardar archivos CSV
bodegas_curated.to_csv("curated/bodegas_curated.csv", index=False)
bodegas_rejects.to_csv("rejects/bodegas_rejects.csv", index=False)